# 03. Answer-generation uncertainty

Generate repeated answers from one fixed evidence set and quantify semantic variability.

In [1]:
from pathlib import Path
from collections import Counter
import json
import math

import numpy as np
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from openai import OpenAI

## 1. Configuration

In [2]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd] + list(cwd.parents):
        if (path / "outputs").exists():
            return path
    return cwd

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR = OUTPUT_DIR / "processed_data"

SAMPLE_MODE = True
RUN_NAME = "sample" if SAMPLE_MODE else "full"

EVIDENCE_DIR = OUTPUT_DIR / "evidence_selection" / RUN_NAME
ANSWER_DIR = OUTPUT_DIR / "answer_generation" / RUN_NAME
ANSWER_DIR.mkdir(parents=True, exist_ok=True)

BACKEND = "ollama" if SAMPLE_MODE else "vllm"

OLLAMA_MODEL = "gemma3:12b"
VLLM_MODEL = "google/gemma-3-12b-it"

MODEL = (
    OLLAMA_MODEL
    if BACKEND == "ollama"
    else VLLM_MODEL
)

VLLM_BASE_URL = "http://localhost:8000/v1"

N_RUNS = 10
TEMPERATURE = 0.7
TOP_P = 0.9
NUM_CTX = 8192

BASE_SEED = 1000
RUN_SEEDS = [BASE_SEED + i for i in range(N_RUNS)]

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CLUSTER_DISTANCE = 0.10
THRESHOLDS_FOR_SENSITIVITY = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
]

RAW_PATH = ANSWER_DIR / "answer_runs.jsonl"
CLUSTER_PATH = ANSWER_DIR / "answer_runs_with_clusters.csv"
SUMMARY_PATH = ANSWER_DIR / "answer_summary.csv"
SENSITIVITY_PATH = ANSWER_DIR / "threshold_sensitivity.csv"

## 2. Shared functions

In [3]:
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def append_jsonl(record, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def parse_ids(value):
    if isinstance(value, list):
        return [str(x) for x in value]
    if pd.isna(value):
        return []
    return [str(x) for x in json.loads(value)]

def selected_evidence(case, selected_ids):
    selected_ids = set(map(str, selected_ids))
    return [
        sentence
        for sentence in case["sentences"]
        if str(sentence["sentence_id"]) in selected_ids
    ]

def build_prompt(case, selected_ids):
    evidence = selected_evidence(case, selected_ids)

    evidence_text = (
        "\n".join(
            f"[{s['sentence_id']}] {s['text']}"
            for s in evidence
        )
        if evidence
        else "No evidence sentences were selected."
    )

    if case["dataset"] == "archehr_qa":
        question_block = f"""
Patient question:
{case["patient_question"]}

Clinician-interpreted question:
{case["clinician_question"]}

Clinical specialty:
{case["clinical_specialty"]}
""".strip()
        register = "professional clinical"
    else:
        question_block = f"""
Question:
{case["question"]}
""".strip()
        register = "professional biomedical"

    return f"""
You are an expert clinical NLP assistant. Generate a concise answer using only the selected evidence provided below.

{question_block}

Selected evidence:
{evidence_text}

Instructions:
- Answer the question using only information in the selected evidence.
- Write in a {register} register.
- Do not use outside medical knowledge or add unsupported information.
- Do not speculate or make inferences beyond what is explicitly supported by the evidence.
- If the evidence is insufficient to fully answer the question, give a faithful answer based only on what is available.
- Limit the answer to 75 words.
- Do not include sentence IDs, citations, or explanations of your reasoning.

Return only the answer text.
""".strip()

def call_ollama(prompt, seed):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "seed": seed,
                "num_predict": 180,
                "num_ctx": NUM_CTX,
            },
        },
        timeout=600,
    )
    response.raise_for_status()

    return " ".join(
        response.json()["response"].strip().split()
    )


def call_vllm(prompt, seed):
    response = requests.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json={
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "seed": seed,
            "max_completion_tokens": 180,
            "stream": False,
        },
        timeout=600,
    )
    response.raise_for_status()

    text = (
        response.json()["choices"][0]["message"]["content"]
    )

    return " ".join(text.strip().split())


def call_model(prompt, seed):
    if BACKEND == "ollama":
        return call_ollama(
            prompt,
            seed=seed,
        )

    if BACKEND == "vllm":
        return call_vllm(
            prompt,
            seed=seed,
        )

    raise ValueError(f"Unknown backend: {BACKEND}")


def check_model_server():
    if BACKEND == "ollama":
        url = "http://localhost:11434/"
    else:
        url = f"{VLLM_BASE_URL}/models"

    response = requests.get(
        url,
        timeout=10,
    )
    response.raise_for_status()
    

def average_pairwise_embedding_distance(answers, model):
    embeddings = model.encode(
        answers,
        normalize_embeddings=True,
    )
    distances = cosine_distances(embeddings)
    upper = distances[np.triu_indices(len(answers), k=1)]
    return float(upper.mean()) if len(upper) else 0.0

def cluster_answers(
    answers,
    embedding_model,
    distance_threshold=CLUSTER_DISTANCE,
):
    if len(answers) == 1:
        return [0], np.array([[0.0]])

    embeddings = embedding_model.encode(
        answers,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    distance_matrix = 1.0 - cosine_similarity(embeddings)

    try:
        clustering = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=distance_threshold,
        )
    except TypeError:
        clustering = AgglomerativeClustering(
            n_clusters=None,
            affinity="precomputed",
            linkage="average",
            distance_threshold=distance_threshold,
        )

    labels = clustering.fit_predict(distance_matrix).tolist()
    return labels, distance_matrix

def semantic_cluster_entropy(labels):
    counts = Counter(labels)
    n = len(labels)

    probabilities = [
        count / n
        for count in counts.values()
    ]
    entropy = -sum(
        p * math.log(p)
        for p in probabilities
        if p > 0
    )

    return entropy / math.log(n) if n > 1 else 0.0

## 3. Load fixed evidence

In [4]:
evidence_summary = pd.read_csv(
    EVIDENCE_DIR / "evidence_summary.csv"
)

evidence_summary["selected_sentence_ids"] = (
    evidence_summary["selected_sentence_ids"].apply(parse_ids)
)

case_files = {
    "bioasq_train": PROCESSED_DIR / "bioasq_train_cases.jsonl",
    "bioasq_test": PROCESSED_DIR / "bioasq_test_cases.jsonl",
    "archehr_train": PROCESSED_DIR / "archehr_train_cases.jsonl",
    "archehr_test": PROCESSED_DIR / "archehr_test_cases.jsonl",
}

active_sets = set(evidence_summary["analysis_set"])

case_by_key = {
    (analysis_set, case["case_id"]): case
    for analysis_set, path in case_files.items()
    if analysis_set in active_sets
    for case in load_jsonl(path)
}

print("Cases from Notebook 02:", len(evidence_summary))

Cases from Notebook 02: 6


## 4. Run repeated answer generation

In [5]:
existing = load_jsonl(RAW_PATH)
done = {
    (row["analysis_set"], row["case_id"], int(row["run_id"]))
    for row in existing
}

try:
    check_model_server()
except Exception as exc:
    raise RuntimeError(
        f"Start {BACKEND} before running repeated answer generation."
    ) from exc

for row in evidence_summary.itertuples():
    case = case_by_key[(row.analysis_set, row.case_id)]

    for run_id in range(N_RUNS):
        key = (row.analysis_set, row.case_id, run_id)
        if key in done:
            continue

        seed = RUN_SEEDS[run_id]

        answer = call_model(
            build_prompt(
                case,
                row.selected_sentence_ids,
            ),
            seed=seed,
        )

        record = {
            "analysis_set": row.analysis_set,
            "case_id": row.case_id,
            "run_id": run_id,
            "seed": seed,
            "answer": answer,
        }

        append_jsonl(record, RAW_PATH)
        done.add(key)

        print(
            row.analysis_set,
            row.case_id,
            run_id + 1,
        )

## 5. Calculate answer-generation uncertainty

In [6]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
runs_df = pd.DataFrame(load_jsonl(RAW_PATH))

cluster_rows = []
summary_rows = []
sensitivity_rows = []

for (analysis_set, case_id), group in runs_df.groupby(
    ["analysis_set", "case_id"],
    sort=False,
):
    group = group.sort_values("run_id").copy()

    if len(group) != N_RUNS:
        raise ValueError(
            f"{case_id}: expected {N_RUNS} answers, "
            f"found {len(group)}"
        )

    answers = group["answer"].tolist()

    labels, _ = cluster_answers(
        answers,
        embedding_model,
    )
    group["cluster_label"] = labels

    for row in group.itertuples():
        cluster_rows.append({
            "analysis_set": analysis_set,
            "case_id": case_id,
            "run_id": int(row.run_id),
            "cluster_label": int(row.cluster_label),
            "answer": row.answer,
        })

    summary_rows.append({
        "analysis_set": analysis_set,
        "case_id": case_id,
        "answer_uncertainty": semantic_cluster_entropy(labels),
        "answer_pairwise_distance": average_pairwise_embedding_distance(
            answers,
            embedding_model,
        ),
        "n_answer_clusters": len(set(labels)),
    })

    for threshold in THRESHOLDS_FOR_SENSITIVITY:
        threshold_labels, _ = cluster_answers(
            answers,
            embedding_model,
            distance_threshold=threshold,
        )

        sensitivity_rows.append({
            "analysis_set": analysis_set,
            "case_id": case_id,
            "threshold": threshold,
            "answer_uncertainty": semantic_cluster_entropy(
                threshold_labels
            ),
            "n_answer_clusters": len(set(threshold_labels)),
        })

answer_runs_with_clusters = pd.DataFrame(cluster_rows)
answer_summary = pd.DataFrame(summary_rows)
threshold_sensitivity = pd.DataFrame(sensitivity_rows)

answer_runs_with_clusters.to_csv(
    CLUSTER_PATH,
    index=False,
)
answer_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)
threshold_sensitivity.to_csv(
    SENSITIVITY_PATH,
    index=False,
)

print("Saved:", SUMMARY_PATH)
display(
    answer_summary.groupby("analysis_set")[
        [
            "answer_uncertainty",
            "answer_pairwise_distance",
            "n_answer_clusters",
        ]
    ].mean()
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: /Users/sangbin/Desktop/Dissertation/Msc_Dissertation/outputs/answer_generation/sample/answer_summary.csv


,answer_uncertainty,answer_pairwise_distance,n_answer_clusters
analysis_set,,,
archehr_train,0.144881,0.055246,1.666667
bioasq_train,0.000000,0.027143,1.000000


## 6. Clustering-threshold sensitivity

In [7]:
# Summarise clustering-threshold sensitivity
sensitivity_summary = (
    threshold_sensitivity
    .groupby(["analysis_set", "threshold"])
    .agg(
        mean_answer_uncertainty=("answer_uncertainty", "mean"),
        median_answer_uncertainty=("answer_uncertainty", "median"),
        zero_uncertainty=(
            "answer_uncertainty",
            lambda x: (x == 0).mean(),
        ),
        mean_clusters=("n_answer_clusters", "mean"),
    )
    .reset_index()
)

display(sensitivity_summary)

,analysis_set,threshold,mean_answer_uncertainty,median_answer_uncertainty,zero_uncertainty,mean_clusters
0,archehr_train,0.05,0.225156,0.217322,0.333333,2.000000
1,archehr_train,0.10,0.144881,0.217322,0.333333,1.666667
2,archehr_train,0.15,0.072441,-0.000000,0.666667,1.333333
3,archehr_train,0.20,0.000000,-0.000000,1.000000,1.000000
4,archehr_train,0.25,0.000000,-0.000000,1.000000,1.000000
5,archehr_train,0.30,0.000000,-0.000000,1.000000,1.000000
6,bioasq_train,0.05,0.092509,-0.000000,0.666667,1.666667
7,bioasq_train,0.10,0.000000,-0.000000,1.000000,1.000000
8,bioasq_train,0.15,0.000000,-0.000000,1.000000,1.000000
9,bioasq_train,0.20,0.000000,-0.000000,1.000000,1.000000
